# Install And Load Libraries

In [ ]:
# Installed all the required libraries.
# !pip install \
# numpy \
# pandas \
# matplotlib \
# seaborn \
# kagglehub \
# scikit-learn \
# imblearn \
# scipy \
# shap

In [ ]:
# Import all the required libraries for the project.
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns

import os

from pathlib import Path

import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (accuracy_score,
                             precision_score,
                             recall_score,
                             f1_score,
                             roc_curve,
                             auc,
                             matthews_corrcoef,
                             confusion_matrix,
                             classification_report)

from scipy.stats import randint, uniform

import shap

# Set Up File Directories

In [ ]:
# Find root directory.
root_name = 'Web_Phish_Project'
base_dir = Path.cwd()

# Set root directory.
if base_dir.name == root_name:
    root = base_dir
elif base_dir.parent.name == root_name:
    root = base_dir.parent
elif base_dir.parent.parent.name == root_name:
    root = base_dir.parent.parent
else:
    print(f'Cannot find root directory: {root_name}.')

print(f'Root folder: {root}')

dataset_dir = root/'saved_data'/'datasets'
saved_models_dir = root/'saved_data'/'trained_models'

# Load And Configure Dataset

In [ ]:
# Load df and display info.
df = pd.read_csv(dataset_dir/'url_dataset.csv')
display(df.head())
display(df.info())

In [ ]:
# Seperate df into features and targets.
x = df.drop('status', axis = 1)
y = df['status']

print(f'Shape of x: {x.shape}.')
print(f'Shape of y: {y.shape}.\n')

# Stratified train-test split.
x_train, x_test, y_train, y_test = train_test_split(x, 
                                                    y,
                                                    test_size = 0.2,
                                                    random_state = 42,
                                                    stratify = y)

# Confirm correct shapes and status distributions.
print(f'Shape of x_train: {x_train.shape}.')
print(f'Shape of y_train: {y_train.shape}.\n')
print(f'Shape of x_test: {x_test.shape}.')
print(f'Shape of y_test: {y_test.shape}.\n')

print(f'Distribution of y_train: {y_train.value_counts()}\n')
print(f'Distribution of y_test: {y_test.value_counts()}')

# Prepare, Tune, And Train Model

In [ ]:
# Create classifier.
rf = RandomForestClassifier(random_state = 42)

# Stratified K-fold setup. 
strat_kfold = StratifiedKFold(n_splits = 4, shuffle = True, random_state = 42)

# Parameter spaces.
rf_params = {'n_estimators': randint(300, 800),
             'criterion': ['gini', 'entropy'],
             'max_depth': randint(10, 60),
             'min_samples_split': randint(2, 40),
             'min_samples_leaf': randint(1, 15),
             'max_features': [None, 'sqrt', 'log2']}

# Search object to find best model configuration.
rf_search = RandomizedSearchCV(rf, 
                               param_distributions = rf_params,
                               n_iter = 50, 
                               scoring = 'recall',
                               n_jobs = -1,
                               refit = True,
                               cv = strat_kfold,
                               random_state = 42)

In [ ]:
# Create directory if not found.
os.makedirs(saved_models_dir, exist_ok = True)

# Set filename and filepath.
filename = 'url_rf'
filepath = os.path.join(saved_models_dir, f'{filename}.pkl')

# Check if file already exists. If not, train search object and save.
if os.path.exists(filepath):
    print(f'{filename}.pkl alrady exists. Skip training.')
else:
    rf_search.fit(x_train, y_train)
    joblib.dump(rf_search, filepath)

In [ ]:
# Load search object.
r_search = joblib.load(filepath)
print(f'{filename} successfuly loaded from {filepath}.')

In [ ]:
# Print best parameters.
print('--- Best parameters ---')
display(r_search.best_params_)

# Print cross valiation score for recall.
print('\n--- Best Cross Validation Recall ---')
print(round(r_search.best_score_, 2))

# Get the best model.
best_rf = r_search.best_estimator_

# Get recall training score on best model.
train_pred = best_rf.predict(x_train)
train_recall = recall_score(y_train, train_pred)

# Print training recall score.
print('\n--- Training Recall On Best Model ---')
print(round(train_recall, 2))

# Get the feature importances for best model.
print('\n--- Most Important Features ---')
feat_names = x_train.columns

# Add feature names column as index.
best_feats_df = pd.DataFrame({'features': feat_names}).set_index('features')
# Add feature importances column.
best_feats_df['feature_values'] = best_rf.feature_importances_

# A series with feature importances sorted by highest values.
best_feats_series = best_feats_df['feature_values'].sort_values(ascending = False)

display(best_feats_series.head(10))

# Testing And Results

In [ ]:
def test_model(model, threshold):
    """
    Test model and save results.

    Parameters
    ----------
    model: RandomForestClassifier
        Random Forest Classifier object to test.
    threshold : float
        A threshold adjustment for predictions.

    Returns
    -------
    num_res : dict
        Contains evaluation metrics from model predictions.
    obj_res : dict
        Contains evaluation objects from model predications.
    """
    num_res = {}
    obj_res = {}
    
    # Get model probability predictions for case: phishing.
    y_proba = model.predict_proba(x_test)[:, 1]
    # Adjust prediction threshold.
    y_pred = (y_proba >= threshold).astype(int)

    # Calculate FPR, TPR, and thresholds.
    fpr, tpr, _ = roc_curve(y_test, y_proba)

    # Populate results dictionaries.
    num_res = {'accuracy': accuracy_score(y_test, y_pred),
               'precision': precision_score(y_test, y_pred, pos_label = 1),
               'recall': recall_score(y_test, y_pred, pos_label = 1),
               'f1': f1_score(y_test, y_pred, pos_label = 1),
               'mcc': matthews_corrcoef(y_test, y_pred),
               'auc': auc(fpr, tpr),
               'threshold': threshold}

    obj_res = {'cm': confusion_matrix(y_test, y_pred),
               'fpr': fpr,
               'tpr': tpr,
               'cr': classification_report(y_test,
                                           y_pred,
                                           target_names = ['Legitimate', 'Phishing'],
                                           output_dict = True)}

    return num_res, obj_res
# End of function.

In [ ]:
# Check predictions at various thresholds.
test_num_dict, _ = test_model(best_rf, 0.5)
test_num_dict2, __ = test_model(best_rf, 0.4)
test_num_dict3, ___ = test_model(best_rf, 0.3)
test_num_dict4, ____ = test_model(best_rf, 0.2)
test_num_dict5, ____ = test_model(best_rf, 0.1)

In [ ]:
# Display various number results.
test_num_series = pd.Series(test_num_dict).round(2)
print(test_num_series, '\n')

test_num_series2 = pd.Series(test_num_dict2).round(2)
print(test_num_series2, '\n')

test_num_series3 = pd.Series(test_num_dict3).round(2)
print(test_num_series3, '\n')

test_num_series4 = pd.Series(test_num_dict4).round(2)
print(test_num_series4, '\n')

test_num_series5 = pd.Series(test_num_dict5).round(2)
print(test_num_series5)

In [ ]:
# Chosen threshold: best trade off between recall and other metrics.
num_dict, obj_dict = test_model(best_rf, 0.4)
num_series = pd.Series(num_dict).round(2)
print(num_series)

In [ ]:
class_names = ['Legit', 'Phish']

# Print confusion matrix heatmap.
plt.figure(figsize = (4, 3))

sns.heatmap(obj_dict['cm'],
           annot = True,
           fmt = '.0f',
           cmap = 'Greens',
           cbar = False,
           linewidths = 1,
           linecolor = 'black',
           xticklabels = class_names,
           yticklabels = class_names)

plt.title('Random Forest Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()

plt.show()
plt.close()

In [ ]:
# Print ROC AUC graph.
plt.figure(figsize = (4, 4))

fpr = obj_dict['fpr']
tpr = obj_dict['tpr']
auc_score = num_dict['auc']
    
plt.plot(fpr, tpr, label = f'Random Forest AUC = {auc_score:.2f}')
plt.plot([0, 1], [0, 1], 'k--')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Random Forest ROC AUC')

plt.legend()
plt.grid()

plt.tight_layout()

plt.show()
plt.close()

In [ ]:
# Print classification report.
cr_df = pd.DataFrame(obj_dict['cr']).T.round(2)
display(cr_df)

# SHAP Analysis

In [ ]:
# Take a random sample amount from testing data for use in shap.
sampled_x_test = x_test.sample(300, random_state = 42)
sampled_indices = sampled_x_test.index

# Create explainer object from best model.
explainer = shap.Explainer(best_rf)

# Create SHAP object from best model and sampled test data.
shap_obj = explainer(sampled_x_test)

In [ ]:
# Find original index positions from x_test for mapping to y_test.
original_index = sampled_indices[0]
original_index2 = sampled_indices[1]
print(f'Original index for row 0 is {original_index}.')
print(f'Original index for row 1 is {original_index2}.\n')

# Double check indices match.
print(f'Label located at row position for first sampled row record: {y_test.index.get_loc(original_index)}.')
print(f'Label located at row position for second sampled row record: {y_test.index.get_loc(original_index2)}.\n')

print(f'Original index for y_test row 1517: {y_test.index[1517]}.')
print(f'Original index for y_test row 1225: {y_test.index[1225]}.\n')

# Find two instances from the sample, one phishing and the other legit.
print('Instance at index 0 belongs to class:', y_test.loc[original_index])
print('Instance at index 1 belongs to class:', y_test.loc[original_index2])

legit_index = 0
phish_index = 1

plot_shap_values = shap_obj[:, :, 1]

print()
shap.plots.beeswarm(plot_shap_values)

print('\n--- Legitimate Record ---')
shap.plots.waterfall(plot_shap_values[legit_index])

print('\n--- Phishing Record ---')
shap.plots.waterfall(plot_shap_values[phish_index])

# Save Best Model

In [ ]:
# Create directory if not found.
os.makedirs(saved_models_dir, exist_ok = True)

# Set filename and filepath.
filename = 'best_url_rf'
filepath = os.path.join(saved_models_dir, f'{filename}.pkl')

# Check if file already exists. If not, save bests model.
if os.path.exists(filepath):
    print(f'{filename}.pkl alrady exists. Skip saving.')
else:
    joblib.dump(best_rf, filepath)

In [ ]:
# End of notebook.